## **Project name: G. Human in the loop machine translation**
#### **team members:**
* Aissat Imane
* Abid Sara
* Ahmane Younes
* Rouainia Feriel
* Hidoussi Takoua
* Bentiba Lina Wafaa

# **Notebook overview**

This notebook contains our approach on building and evaluating a human-in-the-loop (HITL) machine translation pipeline for English/French → Arabic, combining strong pretrained sequence-to-sequence models with task-specific fine-tuning and evaluation.

1. corpus basic preprocessing
2. Models evaluation on our 7k evaluation set.
3. choice of top 3 models based on time and performance.
4. Finetuning code using optimization techniques.
5. Human in the loop pipeline after translation output (preprocessing -> validation -> deduplication -> execution -> monitoring)



#### imports, libraries and such

In [ ]:
import os
import json
import glob
import shutil
import random
import time
from datetime import datetime
from pathlib import Path
import contextlib

import numpy as np
import pandas as pd
import nltk
import evaluate
from tqdm import tqdm

import torch
from torch.utils.data import DataLoader
from torch.optim import AdamW

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    DataCollatorForSeq2Seq,
    get_scheduler,
)

from huggingface_hub import snapshot_download

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
    prepare_model_for_kbit_training,
    PeftModel,
)
import json
import re
import unicodedata
from datetime import datetime, timezone
from typing import Dict, List, Optional, Tuple
import langdetect
import torch
from sacrebleu.metrics import CHRF
from sentence_transformers import SentenceTransformer, util
from transformers import AutoTokenizer, AutoModelForCausalLM
from jsonschema import validate, ValidationError
from huggingface_hub import login
login(token="login_token")
import faiss

from typing import Any

import threading
import shutil
from datetime import datetime, timedelta
from typing import List, Dict, Optional
import warnings
import evaluate

import xml.etree.ElementTree as ET
import string

# **Data pre processing**

This is the preprocessing code that was applied on our data, for each language, language-specific normalization and processing has been applied.

**Accross all languages:**
* Removed transcript markers and annotations, including:
    * Content inside parentheses (applause, laughter, …)
    * Content inside brackets [applause, laughter, …]
* Removed leading and trailing hyphens
* Removed duplicated hyphens inside sentences
* Normalized whitespace (collapsed multiple spaces)
* Removed duplicated punctuation (e.g., !!, ??)

* Filtered out:
    * Empty sentence pairs
    * Very short sentences (length < 3 characters)
    * Enforced strict 1:1 sentence alignment
    * Removed duplicate sentence pairs after alignment

**Arabic-specific preprocessing:**
For Arabic source sentences, we applied additional normalization steps:
* Normalized Arabic characters:
    * Alif variants → ا (أ, إ, آ, ٱ)
    * Ya variants → ي (ى, ئ)

* Normalized Arabic punctuation to Latin equivalents:
  * ، → , ؛ → ; ؟ → ?
  * « » → "
  * ٫ , ． → .

* Converted Arabic digits to Latin digits
* Removed emojis and symbolic Unicode characters
* Removed Latin letters from Arabic text
* Removed Arabic diacritics (tashkīl):
  * Fatha, Damma, Kasra, Shadda, Sukūn, Tanwīn, Tatwīl

* Normalized hyphens and dashes: Unified all dash variants to -
* Collapsed repeated hyphens and duplicated punctuation
* Normalized quotation marks to standard double quotes (")

**English/french preprocessing:**
* Applied Unicode normalization (NFC)
* Normalized apostrophes to a single form (')
* Normalized quotation marks:
* Converted all variants (« » “ ” „) to "
* Normalized hyphens and dashes: Unified all dash variants to -
* Collapsed repeated hyphens
* Removed HTML tags
* Removed control and non-printable Unicode characters

In [ ]:
def clean_text(text):
    """
    Remove transcript markers and formatting artifacts:
    - Content between parentheses: (applause), (laughter), etc.
    - Content between brackets: [applause], [laughter], etc.  
    - Leading/trailing hyphens
    - Duplicate hyphens inside (- -)
    - Extra spaces
    - Duplicated punctuation
    """
    # Remove content between parentheses
    text = re.sub(r'\([^)]*\)', '', text)
    
    # Remove content between brackets
    text = re.sub(r'\[[^\]]*\]', '', text)
    
    # Remove content between French parentheses with spaces: ( Applaudissements )
    text = re.sub(r'\(\s*[^)]*\s*\)', '', text)
    
    # Remove leading/trailing hyphens with optional spaces
    text = re.sub(r'^\s*(?:-+\s*)+', '', text)
    text = re.sub(r'(?:\s*-+)+\s*$', '', text)
    
    # Remove repeated hyphens inside the sentence
    text = re.sub(r'(-\s*){2,}', ' ', text)
    
    # Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    # Remove duplicated punctuation
    text = re.sub(r'([?.!,;:])\1+', r'\1', text)
    
    return text


def normalize_arabic(text):
    # Remove transcript markers and clean artifacts
    text = clean_text(text)

    arabic_replacements = {
        "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا",
        "ى": "ي", "ئ": "ي"
    }
    for orig, repl in arabic_replacements.items():
        text = text.replace(orig, repl)

    arabic_punct = {"،": ",", "؛": ";", "؟": "?", "«": '"', "»": '"', "٫": ".", "．": "."}
    for orig, repl in arabic_punct.items():
        text = text.replace(orig, repl)

    arabic_digits = {"٠": "0", "١": "1", "٢": "2", "٣": "3", "٤": "4",
                     "٥": "5", "٦": "6", "٧": "7", "٨": "8", "٩": "9"}
    for orig, repl in arabic_digits.items():
        text = text.replace(orig, repl)

    # Remove emojis + symbols
    text = re.sub(r'[\U0001F600-\U0001F64F'
                  r'\U0001F300-\U0001F5FF'
                  r'\U0001F680-\U0001F6FF]', '', text)

    # Remove Latin letters (Arabic side only)
    text = re.sub(r'[A-Za-z]', '', text)

    # Remove diacritics
    arabic_diacritics = re.compile("""
                             ّ | َ | ً | ُ | ٌ | ِ | ٍ | ْ | ـ
                         """, re.VERBOSE)
    text = re.sub(arabic_diacritics, '', text)

    # Normalize hyphens
    hyphens = ["–", "—", "ـ", "−", "_", "\u2011"]
    for h in hyphens:
        text = text.replace(h, "-")
    text = re.sub(r'-+', '-', text)

    # Normalize quotes
    quotes = ["„", "“", "”", "«", "»"]
    for q in quotes:
        text = text.replace(q, '"')

    return text


def normalize_latin(text):
    # Remove transcript markers and clean artifacts
    text = clean_text(text)

    # Decode escaped quotes/backslashes
    text = text.replace('\\"', '"').replace('\\\\', '\\')

    # Unicode normalization
    text = unicodedata.normalize("NFC", text)

    # Normalize apostrophes
    apostrophes = ["'", "`", "´", "ʼ"]
    for a in apostrophes:
        text = text.replace(a, "'")

    # Normalize quotation marks
    quote_map = {
        "«": '"', "»": '"',
        "“": '"', "”": '"', "„": '"'
    }
    for q, repl in quote_map.items():
        text = text.replace(q, repl)

    # Normalize hyphens/dashes
    hyphens = ["–", "—", "−", "-", "‒", "_", "\u2011"]
    for h in hyphens:
        text = text.replace(h, "-")
    text = re.sub(r'-+', '-', text)

    # Remove HTML tags
    text = re.sub(r'<[^>]+>', '', text)

    # Remove control characters
    text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != "C")

    return text



def build_json(source_file, target_file, output_file, src_lang="ar", tgt_lang="fr"):
    
    # Load ALL sentences
    print("Loading sentence files")
    with open(source_file, 'r', encoding='utf-8') as f:
        source_sentences = [normalize_arabic(line.strip()) for line in f]

    with open(target_file, 'r', encoding='utf-8') as f:
        target_sentences = [normalize_latin(line.strip()) for line in f]

    print(f"Loaded {len(source_sentences):,} Arabic sentences")
    print(f"Loaded {len(target_sentences):,} French/English sentences")

    # Create direct 1:1 alignments for ALL sentences
    print("\n=== Creating direct 1:1 alignments ===")
    data = []
    empty_pairs = 0
    short_pairs = 0
    
    # Use the minimum length to ensure we don't go out of bounds
    min_length = min(len(source_sentences), len(target_sentences))
    
    for i in range(min_length):
        src_sent = source_sentences[i]
        tgt_sent = target_sentences[i]
        
        # Skip empty sentences
        if not src_sent or not tgt_sent:
            empty_pairs += 1
            continue
            
        # Skip very short sentences (likely noise)
        if len(src_sent.strip()) < 3 or len(tgt_sent.strip()) < 3:
            short_pairs += 1
            continue
            
        data.append({
            "source_language": src_lang,
            "target_language": tgt_lang,
            "source_text": src_sent,
            "target_text": tgt_sent
        })

    print(f"Created {len(data):,} direct 1:1 alignments")
    print(f"Skipped {empty_pairs:,} empty sentence pairs")
    print(f"Skipped {short_pairs:,} very short sentence pairs")
    print(f"Utilization: {len(data):,} / {min_length:,} possible pairs ({len(data)/min_length*100:.1f}%)")

    # Remove any remaining duplicates
    print("\n=== Removing duplicates ===")
    unique_data = []
    seen_pairs = set()
    duplicates_removed = 0
    
    for item in data:
        pair_key = (item["source_text"], item["target_text"])
        if pair_key not in seen_pairs:
            seen_pairs.add(pair_key)
            unique_data.append(item)
        else:
            duplicates_removed += 1

    print(f"Final unique pairs: {len(unique_data):,}")
    print(f"Duplicates removed: {duplicates_removed}")

    # Save JSON
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(unique_data, f, ensure_ascii=False, indent=2)

    print(f"\n=== FINAL RESULT ===")
    print(f"Saved {len(unique_data):,} aligned sentences to {output_file}")



# First, model evaluation

Before going to the finetuning step, 7k parallel pairs from the overall collected data were kept as an evaluation set.
Many models were inferred on the pairs of this set and their performance was assessed using multiple metrics like BLEU, METEOR, COMET, Chrf++ and TRF.

##### **The evaluated models:**
* Facebook NLLB 200 3.3B
* Facebook NLLB 200 1.3B
* MADLAD 3.3B
* LMT 600 1.7B
* Gemma 2-2B
* Facebook NLLB-200-Distilled-600M
* Qwen-3-4B
* Facebook M2M-100-1.2B
* Helsinki opus-mt-en-ar

Then, taking into consideration the performance of each model in EN -> AR and FR -> AR directions, the top 3 were chosen to be finetuned further.


### **some samples evaluation codes**

In [ ]:
# QWEN 3 4B
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_SAVE_FOLDER = "/kaggle/working/qwen_model_4b"
DATA_FOLDER = "/kaggle/input/evaluation-set"
OUTPUT_FOLDER = "/kaggle/working/translations"
BATCH_SIZE = 32  
MAX_NEW_TOKENS = 128
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(MODEL_SAVE_FOLDER, exist_ok=True)
os.makedirs(OUTPUT_FOLDER, exist_ok=True)


# loading model
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, 
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

if os.listdir(MODEL_SAVE_FOLDER):
    print("Loading saved 4-bit model")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_SAVE_FOLDER, padding_side='left')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_SAVE_FOLDER,
        device_map="auto",
        quantization_config=bnb_config,
        torch_dtype=torch.float16
    )
else:
    print("Downloading & saving 4-bit model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, padding_side='left')
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
        torch_dtype=torch.float16
    )
    tokenizer.save_pretrained(MODEL_SAVE_FOLDER)
    model.save_pretrained(MODEL_SAVE_FOLDER)

# Set pad token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model.eval()


# helper function
def get_text_to_translate(rec, src_lang, tgt_lang):
    s_lang = rec.get("source_language", "").lower()
    t_lang = rec.get("target_language", "").lower()
    if s_lang == src_lang and t_lang == tgt_lang:
        return rec.get("source_text", "")
    if s_lang == tgt_lang and t_lang == src_lang:
        return rec.get("target_text", "")
    return rec.get("source_text", "")


# batch translation for optimization purposes
def translate_batch(texts, src, tgt, batch_size=BATCH_SIZE):
    translations = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Batches", unit="batch"):
        batch = texts[i:i+batch_size]
        
        # adapted prompting for qwen
        prompts = [f"""You are a professional translator. Translate the following text from {src} to {tgt}.

CRITICAL REQUIREMENTS:
- Output ONLY the direct translation
- Do NOT include any explanations, notes, or commentary
- Do NOT write phrases like "Here is the translation:", "Translation:", "The translation is:", etc.
- Do NOT repeat or reference the source text
- Do NOT add introductory or concluding remarks
- Do NOT include any meta-information about the translation process
- Provide the translated text and absolutely nothing else
- DO NOT REPEAT THE SENTENCE MULTIPLE TIMES LIKE THIS: The Council holds its regular sessions every two years at the United Nations Office in Nairobi. During these sessions, the Council reviews the United Nations Office's work programme covering a two-year period, its budget for the United Nations Office and human settlements, as well as the operational activities carried out by the United Nations Office. The Council holds its regular sessions every two years at the United Nations Office in Nairobi. During these sessions, the Council reviews the United Nations Office's work programme covering a two-year period, its budget for the United Nations Office and human settlements, as well as the operational activities carried out by the United Nations Office. The Council holds its
- DO NOT WRITE ANYTHING ELSE JUST THE TRANSLATED SENTENCE, example don't do this There is no democracy worthy of the name that lacks transparency, but here transparency is one-way openness, and giving the steering wheel without a brake was never the promise of democracy to its citizens. \n\nThe translation is complete. Final output. No further text. No meta-information. No commentary. Only the direct translation. Only the translated text. Only the final output. Only the direct translation. Only the translated text. Only the final output. Only the direct translation. Only the translated text. Only the final output. Only the direct translation. Only the translated text. Only the final output. Only the direct translation. Only the translated text.


Source text:
{text}

Translated text:""" for text in batch]
        
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=512  
        ).to(DEVICE)
        
        with torch.no_grad(), torch.cuda.amp.autocast():  
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,  
                use_cache=True, 
                pad_token_id=tokenizer.pad_token_id,
                num_beams=1,  
            )
        
        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True)
        
        # Extract translations
        for full, prompt in zip(decoded, prompts):
            if full.startswith(prompt):
                translations.append(full[len(prompt):].strip())
            else:
                translations.append(full.strip())
    
    return translations
# translation with checkpoints kept 
def translate_file(file_path, src_lang, tgt_lang):
    print(f"\nTranslating {os.path.basename(file_path)} ({src_lang} → {tgt_lang})")
    
    out_path = os.path.join(
        OUTPUT_FOLDER,
        os.path.basename(file_path).replace(".jsonl", "_translated.jsonl")
    )
    
    # Resume from checkpoint if exists
    if os.path.exists(out_path):
        print(f"Found existing output. Skipping {os.path.basename(file_path)}")
        return
    
    # Load JSONL
    with open(file_path, 'r', encoding='utf-8') as f:
        records = [json.loads(line) for line in f]
    
    print(f"Total records: {len(records)}")
    
    texts = [get_text_to_translate(rec, src_lang, tgt_lang) for rec in records]
    
    # Translate
    outputs = translate_batch(texts, src_lang, tgt_lang)
    
    # Attach hypotheses
    for rec, hyp in zip(records, outputs):
        rec["hypothesis"] = hyp
    
    # Save
    with open(out_path, "w", encoding="utf-8") as f:
        for rec in records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    
    print(f"Saved: {out_path}")
    
    # Clear cache
    torch.cuda.empty_cache()


# running translations of all files
dataset_files = {
    "AR2EN.jsonl": ("ar", "en"),
    "AR2FR.jsonl": ("ar", "fr"),
    "FR2AR.jsonl": ("fr", "ar"), 
    "EN2FR.jsonl": ("en", "ar")
}
for fname, (src, tgt) in dataset_files.items():
    path = os.path.join(DATA_FOLDER, fname)
    if os.path.exists(path):
        translate_file(path, src, tgt)
    else:
        print(f"Missing file: {path}")

In [ ]:
## MADLAD 3.3B evaluation
from transformers import T5ForConditionalGeneration, T5Tokenizer

# Loading MADLAD-400-3B-MT and tokenizer
model_name = "google/madlad400-3b-mt"
tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(
    model_name,
    torch_dtype=torch.float16
)
model = model.to("cuda")  #

# madlad adapted translation function, since it is a tag driven model we define the tags based on target languages
def translate_madlad(text, target_language):
    lang_map = {"English": "<2en>", "French": "<2fr>", "Arabic": "<2ar>"}
    target_token = lang_map[target_language]
    prompt = f"{target_token} {text}"
    
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs,
        max_new_tokens=200,
        do_sample=False
    )
    res = tokenizer.decode(out[0], skip_special_tokens=True)
    return res.strip()


files_and_targets = {
    "AR2EN.jsonl": "English",
    "AR2FR.jsonl": "French",
    "EN2AR.jsonl": "Arabic",
    "FR2AR.jsonl": "Arabic"
}

# an example code to translate from english to arabic
data_path = "evaluation-dataset/"
file_name = "EN2AR.jsonl"
target_lang = "Arabic"
chunk_size = 500 

pairs = []

# Load dataset
with open(data_path + file_name, "r", encoding="utf-8") as f:
    for line in f:
        obj = json.loads(line)
        pairs.append({"source_text": obj["source_text"], "target_text": obj["target_text"]})

print(f"Evaluating {file_name} ({len(pairs)} samples)")

preds = []
refs = []

num_chunks = math.ceil(len(pairs) / chunk_size)

for i in range(num_chunks):
    start = i * chunk_size
    end = min((i + 1) * chunk_size, len(pairs))
    chunk = pairs[start:end]
    
    print(f"\nProcessing chunk {i+1}/{num_chunks} ({len(chunk)} samples)")

    chunk_preds = []
    chunk_refs = []
    
    for sample in chunk:
        src = sample["source_text"]
        tgt = sample["target_text"]
        
        pred = translate_madlad(src, target_lang) 
        refs.append(tgt)
        preds.append(pred)
        chunk_refs.append(tgt)
        chunk_preds.append(pred)
    # saving each chunk as checkpoint
    chunk_file = f"prediction_{file_name}_chunk{i+1}.json"
    with open(chunk_file, "w", encoding="utf-8") as f:
        json.dump(chunk_preds, f, ensure_ascii=False, indent=2)
    print(f"Saved chunk {i+1} predictions to {chunk_file}")


# matrics computation

bleu = evaluate.load("bleu")
print("BLEU:", bleu.compute(predictions=preds, references=refs))
meteor = evaluate.load("meteor")
print("METEOR:", meteor.compute(predictions=preds, references=refs))
ter = evaluate.load("ter")
print("TER:", ter.compute(predictions=preds, references=refs))
chrf = evaluate.load("chrf")
print("ChrF++:", chrf.compute(predictions=preds, references=refs))


### **Final evaluation results for all evaluated models**


| Model             | BLEU  | METEOR | TER    | chrF++ | COMET |
|------------------|-------|--------|--------|--------|--------|
| NLLB-200-600M    | 2.73  | 7.61   | 113.94 | 8.00   | 51.82  |
| MBART-50 (2.1B)  | 2.00  | 5.16   | 118.03 | 3.41   | 46.35  |
| Qwen2-1.5B       | 0.42  | 6.35   | 561.61 | 7.43   | 38.42  |
| Qwen-3-4B        | 8.56  | 26.33  | 77.67  | 37.48  | 64.45  |
| M2M-100          | —     | 28.05  | 81.45  | 42.47  | —      |
| Gemma-2-2B       | 9.76  | 28.59  | 85.54  | 36.78  | 70.83  |
| MADLAD-3.3B      | 27.54 | 45.09  | 63.42  | 57.82  | 81.76  |
| NLLB-3.3B        | 21.24 | 41.45  | 69.01  | 48.58  | 80.29  |
| NLLB-1.3B        | —     | 40.30  | 70.85  | 51.20  | 76.60  |
| Helsinki-NLP     | 40.22 | 67.10  | 42.98  | —      | —      |


# Second, finetuning the top 3 models

After analyzing evaluation results, these 3 models were chosen for finetuning
* Facebook NLLB 200 3.3B
* Facebook NLLB 200 1.3B
* Facebook M2M 100 1.2B

however, due to time and resources constraints, only the NLLB 1.3B and NLLB 3.3B were successfully finetuned. As an alternative, Helsinki opus-mt-en-ar was used due to its good evaluation results and time it took to generate the translations.


For optimization purposes, both configurations employed QLoRA methodology, enabling efficient adaptation of large multilingual models while maintaining high translation quality

### Finetuning codes

#### **Finetuning NLLB 200 3.3B**

In [ ]:
def finetune_nllb_33b():
    ### Finetuning NLLB 200 3.3B
    warnings.filterwarnings("ignore")
    os.environ['PYTHONWARNINGS'] = 'ignore'

    # cache setup
    HF_CACHE = "/dev/shm/hf_cache"
    os.environ["HF_HOME"] = HF_CACHE
    os.environ["TRANSFORMERS_CACHE"] = f"{HF_CACHE}/transformers"
    os.environ["HF_DATASETS_CACHE"] = f"{HF_CACHE}/datasets"
    os.environ["HF_EVALUATE_CACHE"] = f"{HF_CACHE}/evaluate"

    for path in [os.environ["TRANSFORMERS_CACHE"], os.environ["HF_DATASETS_CACHE"],
                 os.environ["HF_EVALUATE_CACHE"], f"{HF_CACHE}/model_offload"]:
        os.makedirs(path, exist_ok=True)

    # gpu usage optimization
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.set_float32_matmul_precision("high")

    DTYPE = torch.bfloat16 if torch.cuda.is_available() else torch.float16
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # configurations
    MODEL_NAME = "facebook/nllb-200-3.3B"
    MODEL_PATH = "/dev/shm/nllb_3.3b_model"
    OUTPUT_DIR = "/dev/shm/nllb_3.3b_output"
    TRANSLATIONS_DIR = "/dev/shm/nllb_3.3b_translations"

    TRAIN_FILE = "/dev/shm/chunks/train.jsonl"
    VALID_FILE = "/dev/shm/chunks/validation.jsonl"
    TEST_FILE = "/dev/shm/chunks/test.jsonl"

    MAX_SOURCE_LEN = 256
    MAX_TARGET_LEN = 256

    BATCH_SIZE = 4
    GRAD_ACCUM_STEPS = 8
    LR = 3e-4
    EPOCHS = 3
    SUBSET_RATIO = 0.2
    WARMUP_RATIO = 0.1
    SAVE_STEPS = 1000

    LORA_R = 32
    LORA_ALPHA = 64
    LORA_DROPOUT = 0.05

    SRC_LANG_MAP = {"en": "eng_Latn", "fr": "fra_Latn", "ar": "arb_Arab"}
    TGT_LANG = SRC_LANG_MAP

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(TRANSLATIONS_DIR, exist_ok=True)

    # Loading the NLLB 200 3.3B model
    print("LOADING NLLB-3.3B")

    if not os.path.exists(MODEL_PATH) or len(os.listdir(MODEL_PATH)) < 5:
        print("Downloading the model")
        snapshot_download(
            repo_id=MODEL_NAME,
            local_dir=MODEL_PATH,
            local_dir_use_symlinks=False,
            resume_download=True
        )

    print("Loading tokenizer")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
    tokenizer.pad_token = tokenizer.eos_token

    print("Loading model with 4-bit quantization")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    # check already existing checkpoints to take it from there
    checkpoint_dirs = glob.glob(f"{OUTPUT_DIR}/checkpoint-*")
    RESUME_CHECKPOINT = None

    if checkpoint_dirs:
        checkpoint_dirs.sort(key=lambda x: int(x.split("-")[-1]))
        RESUME_CHECKPOINT = checkpoint_dirs[-1]
        print(f"\n{'='*60}")
        print(f"Existing checkpoint: {RESUME_CHECKPOINT}")
        print(f"{'='*60}")

    # setting up lora
    print("lora set up starting")
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=LORA_R,
        lora_alpha=LORA_ALPHA,
        lora_dropout=LORA_DROPOUT,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "wi", "wo"],
        bias="none",
    )

    if RESUME_CHECKPOINT:
        print(f"Loading LoRA adapters from checkpoint")
        model = PeftModel.from_pretrained(model, RESUME_CHECKPOINT, is_trainable=True)
        # Ensuring that lora parameters are trainable
        for name, param in model.named_parameters():
            if 'lora' in name.lower():
                param.requires_grad = True
    else:
        print(f"Initializing new LoRA adapters...")
        model = get_peft_model(model, lora_config)

    model.print_trainable_parameters()

    # function for preprocessing of input data
    def preprocess_text(example):
        try:
            src_text = example.get("source_text", "").strip()
            tgt_text = example.get("target_text", "").strip()
            src_lang = example.get("source_language")
            tgt_lang = example.get("target_language")

            if not src_text or not tgt_text or src_lang not in SRC_LANG_MAP or tgt_lang not in TGT_LANG:
                example["_remove"] = True
                return example

            tokenizer.src_lang = SRC_LANG_MAP[src_lang]

            model_inputs = tokenizer(src_text, truncation=True, max_length=MAX_SOURCE_LEN)
            labels = tokenizer(tgt_text, truncation=True, max_length=MAX_TARGET_LEN - 1)

            tgt_lang_id = tokenizer.convert_tokens_to_ids(TGT_LANG[tgt_lang])

            example["input_ids"] = model_inputs["input_ids"]
            example["attention_mask"] = model_inputs.get("attention_mask", [1] * len(model_inputs["input_ids"]))
            example["labels"] = [tgt_lang_id] + labels["input_ids"]
            example["_remove"] = False
            return example
        except:
            example["_remove"] = True
            return example

    # loading input data
    print("LOADING DATA")

    def load_jsonl(path):
        data = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line))
        return data

    train_data = load_jsonl(TRAIN_FILE)
    valid_data = load_jsonl(VALID_FILE)

    print(f"Train: {len(train_data):,} samples")
    print(f"Valid: {len(valid_data):,} samples")

    # application of gradient fixes
    model.enable_input_require_grads()

    def make_inputs_require_grad(module, input, output):
        output.requires_grad_(True)

    model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

    if model.get_output_embeddings() is not None:
        model.get_output_embeddings().register_forward_hook(make_inputs_require_grad)

    if hasattr(model, "base_model"):
        model.base_model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

    model.train()
    print("Gradient hooks applied")

    # optimizer and scheduler setup
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,
        betas=(0.9, 0.95),
        weight_decay=0.01,
        eps=1e-8,
    )

    subset_size = int(len(train_data) * SUBSET_RATIO)
    steps_per_epoch = subset_size // (BATCH_SIZE * GRAD_ACCUM_STEPS)
    total_steps = steps_per_epoch * EPOCHS

    print(f"raining samples per epoch: {subset_size:,}")
    print(f"Steps per epoch: {steps_per_epoch:,}")
    print(f"Total training steps: {total_steps:,}")

    scheduler = get_scheduler(
        "cosine",
        optimizer=optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8,
    )

    # calculating starting point from last checkpoint stopped at
    global_step = 0
    start_epoch = 0

    if RESUME_CHECKPOINT:
        global_step = int(RESUME_CHECKPOINT.split("-")[-1])
        start_epoch = global_step // steps_per_epoch
        print(f"\nResuming from step: {global_step:,}")
        print(f"Starting from epoch: {start_epoch + 1}")

        # Fast-forward scheduler to match checkpoint
        for _ in range(global_step):
            scheduler.step()
        print(f"Scheduler synced to step {global_step:,}")

    # training
    print("training started")

    training_start = time.time()

    for epoch in range(start_epoch, EPOCHS):
        print(f"\n{'='*60}")
        print(f"EPOCH {epoch+1}/{EPOCHS}")
        print(f"{'='*60}")

        # Quick filtering and tokenization
        sampled = []
        for item in random.sample(train_data, subset_size):
            if (item.get("source_text") and item.get("target_text") and
                item["source_language"] in SRC_LANG_MAP and item["target_language"] in TGT_LANG):

                tokenizer.src_lang = SRC_LANG_MAP[item["source_language"]]

                model_inputs = tokenizer(
                    item["source_text"],
                    truncation=True,
                    max_length=MAX_SOURCE_LEN,
                )

                labels = tokenizer(
                    item["target_text"],
                    truncation=True,
                    max_length=MAX_TARGET_LEN - 1,
                )

                tgt_lang_id = tokenizer.convert_tokens_to_ids(TGT_LANG[item["target_language"]])

                sampled.append({
                    "input_ids": model_inputs["input_ids"],
                    "attention_mask": model_inputs.get("attention_mask", [1] * len(model_inputs["input_ids"])),
                    "labels": [tgt_lang_id] + labels["input_ids"]
                })

        train_ds = Dataset.from_list(sampled)
        print(f"Dataset ready: {len(train_ds):,} samples")

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=data_collator,
            num_workers=2,
            pin_memory=True,
        )

        optimizer.zero_grad()
        epoch_loss = 0.0

        for step, batch in enumerate(train_loader):
            try:
                batch = {k: v.to(DEVICE) for k, v in batch.items()}

                with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                    outputs = model(**batch)
                    loss = outputs.loss / GRAD_ACCUM_STEPS

                loss.backward()
                epoch_loss += loss.item() * GRAD_ACCUM_STEPS

                if (step + 1) % GRAD_ACCUM_STEPS == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

                if step % 50 == 0:
                    torch.cuda.empty_cache()
                    import gc
                    gc.collect()

                if global_step % 100 == 0 and global_step > 0:
                    lr = scheduler.get_last_lr()[0]
                    avg_loss = epoch_loss / (step + 1)
                    print(f"Step {global_step:7,d} | Loss: {avg_loss:.4f} | LR: {lr:.2e}")

                # Saving a checkpoint every 1000 step
                if global_step % SAVE_STEPS == 0 and (step + 1) % GRAD_ACCUM_STEPS == 0 and global_step > 0:
                    ckpt_path = f"{OUTPUT_DIR}/checkpoint-{global_step}"
                    model.save_pretrained(ckpt_path)
                    tokenizer.save_pretrained(ckpt_path)
                    print(f"Checkpoint saved → {ckpt_path}")

                    # keeping 3 last checkpoints only
                    all_checkpoints = sorted(
                        glob.glob(f"{OUTPUT_DIR}/checkpoint-*"),
                        key=lambda x: int(x.split("-")[-1])
                    )
                    if len(all_checkpoints) > 3:
                        for old_ckpt in all_checkpoints[:-3]:
                            shutil.rmtree(old_ckpt)
                            print(f"Removed old checkpoint: {old_ckpt}")

                    torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e):
                    print(f"OOM at step {global_step}, clearing cache and continuing...")
                    torch.cuda.empty_cache()
                    import gc
                    gc.collect()
                    continue
                else:
                    raise e

        print(f"Epoch {epoch+1} done | Avg loss: {epoch_loss / len(train_loader):.4f}")

    # saving final model
    final_path = f"{OUTPUT_DIR}/final_model"
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)

    print(f"Total time: {(time.time() - training_start)/3600:.2f} hours")
    print(f"Final model: {final_path}")
    print(f"Final step: {global_step:,}")


#### **Finetuning NLLB 200 1.3B**

In [ ]:
def finetune_nllb_13b():
    os.environ['PYTHONWARNINGS'] = 'ignore'

    # HF Cache Setup
    HF_CACHE = "/dev/shm/hf_cache"
    os.environ["HF_HOME"] = HF_CACHE
    os.environ["TRANSFORMERS_CACHE"] = f"{HF_CACHE}/transformers"
    os.environ["HF_DATASETS_CACHE"] = f"{HF_CACHE}/datasets"
    os.environ["HF_EVALUATE_CACHE"] = f"{HF_CACHE}/evaluate"
    os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
    os.environ["HF_HUB_DISABLE_XET"] = "1"

    for path in [
        os.environ["TRANSFORMERS_CACHE"],
        os.environ["HF_DATASETS_CACHE"],
        os.environ["HF_EVALUATE_CACHE"],
        f"{HF_CACHE}/model_offload",
        "/dev/shm/nllb_model"
    ]:
        os.makedirs(path, exist_ok=True)

    # safetensors installation
    try:
        import safetensors
    except ImportError:
        import subprocess
        subprocess.run(["pip", "install", "-q", "safetensors"], check=True)
        import safetensors

    # optimization for school gpu usage
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True  
    torch.set_float32_matmul_precision("high")

    USE_BF16 = torch.cuda.is_available()
    DTYPE = torch.bfloat16 if USE_BF16 else torch.float16
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

    # configurations of NLLB 200 1.3B
    MODEL_NAME = "facebook/nllb-200-1.3B"
    MODEL_PATH = "/dev/shm/nllb_model"
    OUTPUT_DIR = "/dev/shm/nllb_qlora_output"
    TRANSLATIONS_DIR = "/dev/shm/nllb_translations"

    TRAIN_FILE = "/dev/shm/chunks/train.jsonl"
    VALID_FILE = "/dev/shm/chunks/validation.jsonl"
    TEST_FILE = "/dev/shm/chunks/test.jsonl"

    MAX_SOURCE_LEN = 256
    MAX_TARGET_LEN = 256

    # hyperparameters setup
    BATCH_SIZE = 16               
    GRAD_ACCUM_STEPS = 4         
    EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRAD_ACCUM_STEPS  

    LR = 5e-4                    
    WARMUP_STEPS = 2000          
    EPOCHS = 15                  
    SUBSET_RATIO = 0.7           

    # Learning rate schedule parameters
    LR_SCHEDULER_TYPE = "cosine"  
    WEIGHT_DECAY = 0.01           

    SRC_LANG_MAP = {"en": "eng_Latn", "fr": "fra_Latn", "ar": "arb_Arab"}
    TGT_LANG = SRC_LANG_MAP

    os.makedirs(OUTPUT_DIR, exist_ok=True)
    os.makedirs(TRANSLATIONS_DIR, exist_ok=True)

    # downloading model if not yet
    if not os.path.exists(MODEL_PATH) or len(os.listdir(MODEL_PATH)) < 5:
        snapshot_download(
            repo_id=MODEL_NAME,
            local_dir=MODEL_PATH,
            local_dir_use_symlinks=False,
            resume_download=True
        )

    # loading tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH, use_fast=False)
    tokenizer.pad_token = tokenizer.eos_token

    # loading model with quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        MODEL_PATH,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=DTYPE,
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    # Lora setup
    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=16,                    
        lora_alpha=32,           
        lora_dropout=0.05,       
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
        bias="none",            
    )
    model = get_peft_model(model, lora_config)

    # preprocessing function defintion
    def preprocess(example):
        src_lang = SRC_LANG_MAP[example["source_language"]]
        tgt_lang = TGT_LANG[example["target_language"]]
        tokenizer.src_lang = src_lang

        inputs = tokenizer(
            example["source_text"],
            truncation=True,
            max_length=MAX_SOURCE_LEN,
        )

        with tokenizer.as_target_tokenizer():
            labels = tokenizer(
                example["target_text"],
                truncation=True,
                max_length=MAX_TARGET_LEN,
            )

        inputs["labels"] = labels["input_ids"]
        inputs["forced_bos_token_id"] = tokenizer.convert_tokens_to_ids(tgt_lang)
        return inputs

    # metrics loading
    bleu = evaluate.load("sacrebleu")
    meteor = evaluate.load("meteor")
    ter = evaluate.load("ter")
    chrf = evaluate.load("chrf")
    comet = evaluate.load("comet")

    def compute_metrics(preds, refs, srcs):
        return {
            "BLEU": bleu.compute(predictions=preds, references=[[r] for r in refs])["score"],
            "METEOR": meteor.compute(predictions=preds, references=refs)["meteor"],
            "TER": ter.compute(predictions=preds, references=[[r] for r in refs])["score"],
            "chrF++": chrf.compute(predictions=preds, references=[[r] for r in refs], word_order=2)["score"],
            "COMET": comet.compute(predictions=preds, references=refs, sources=srcs)["mean_score"],
        }

    clear_output(wait=True)

    print("training started")

    # hyperparameters setup
    WARMUP_RATIO = 0.1
    EVAL_STEPS = 500
    SAVE_STEPS = 1000

    print(f"Batch size: {BATCH_SIZE}")
    print(f"Gradient accumulation: {GRAD_ACCUM_STEPS}")
    print(f"Effective batch size: {BATCH_SIZE * GRAD_ACCUM_STEPS}")
    print(f"Learning rate: {LR}")
    print(f"Epochs: {EPOCHS}")
    print(f"Subset ratio: {SUBSET_RATIO}")

    # laoding data
    def load_jsonl(path):
        data = []
        with open(path, 'r', encoding='utf-8') as f:
            for line in f:
                data.append(json.loads(line))
        return data

    train_data = load_jsonl(TRAIN_FILE)
    valid_data = load_jsonl(VALID_FILE)
    test_data = load_jsonl(TEST_FILE)

    print(f"Train: {len(train_data)} samples")
    print(f"Valid: {len(valid_data)} samples")
    print(f"Test: {len(test_data)} samples")

    # saving translations
    def save_translations(preds, refs, srcs, split, step):
        ts = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = f"{TRANSLATIONS_DIR}/{split}_step{step}_{ts}.jsonl"
        with open(path, "w", encoding="utf-8") as f:
            for s, p, r in zip(srcs, preds, refs):
                json.dump({"source": s, "prediction": p, "reference": r}, f, ensure_ascii=False)
                f.write("\n")
        print(f"Saved translations → {path}")

    # optimizing with lora parameters
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=LR,
        betas=(0.9, 0.95),
        weight_decay=0.01,
    )

    # Calculate total steps
    subset_size = int(len(train_data) * SUBSET_RATIO)
    steps_per_epoch = subset_size // (BATCH_SIZE * GRAD_ACCUM_STEPS)
    total_steps = steps_per_epoch * EPOCHS

    print(f"Total training steps: {total_steps}")
    print(f"Warmup steps: {int(WARMUP_RATIO * total_steps)}")

    scheduler = get_scheduler(
        "linear",
        optimizer=optimizer,
        num_warmup_steps=int(WARMUP_RATIO * total_steps),
        num_training_steps=total_steps,
    )

    data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

    def preprocess_text(example):
        """
        FIXED: Does NOT use as_target_tokenizer() which causes NoneType errors.
        For NLLB, we prepend the target language token to labels manually.
        """
        try:
            src_text = example.get("source_text", "").strip()
            tgt_text = example.get("target_text", "").strip()
            src_lang = example.get("source_language")
            tgt_lang = example.get("target_language")

            # Validation
            if not src_text or not tgt_text or src_lang not in SRC_LANG_MAP or tgt_lang not in TGT_LANG:
                example["input_ids"] = [0]
                example["attention_mask"] = [0]
                example["labels"] = [0]
                example["_remove"] = True
                return example

            # Set source language
            tokenizer.src_lang = SRC_LANG_MAP[src_lang]

            # Tokenize source
            model_inputs = tokenizer(
                src_text,
                truncation=True,
                max_length=MAX_SOURCE_LEN,
            )

            # Tokenize target (WITHOUT as_target_tokenizer!)
            labels = tokenizer(
                tgt_text,
                truncation=True,
                max_length=MAX_TARGET_LEN - 1,  
            )

            # Safety check
            if not model_inputs.get("input_ids") or not labels.get("input_ids"):
                example["input_ids"] = [0]
                example["attention_mask"] = [0]
                example["labels"] = [0]
                example["_remove"] = True
                return example

            # Get target language token and prepend to labels
            tgt_lang_token = TGT_LANG[tgt_lang]
            tgt_lang_id = tokenizer.convert_tokens_to_ids(tgt_lang_token)
            labels_with_lang = [tgt_lang_id] + labels["input_ids"]

            # Add tokenized fields
            example["input_ids"] = model_inputs["input_ids"]
            example["attention_mask"] = model_inputs.get("attention_mask", [1] * len(model_inputs["input_ids"]))
            example["labels"] = labels_with_lang
            example["_remove"] = False

            return example

        except Exception as e:
            print(f"Warning: Error processing sample: {e}")
            example["input_ids"] = [0]
            example["attention_mask"] = [0]
            example["labels"] = [0]
            example["_remove"] = True
            return example

    # training process

    model.train()
    global_step = 0
    training_start = time.time()
    best_bleu = 0.0

    for epoch in range(EPOCHS):
        epoch_start = time.time()
        print(f"EPOCH {epoch+1}/{EPOCHS}")

        subset_size = int(len(train_data) * SUBSET_RATIO)
        sampled = random.sample(train_data, subset_size)
        print(f"Training on {len(sampled)} samples this epoch")

        train_ds = Dataset.from_list(sampled)

        # Checking if data is already preprocessed
        if "input_ids" in train_ds.column_names and "labels" in train_ds.column_names:
            print("Data is already tokenized, using directly")
            train_ds = train_ds.filter(
                lambda x: 
                    "input_ids" in x 
                    and "labels" in x 
                    and x["input_ids"] is not None 
                    and x["labels"] is not None
                    and len(x["input_ids"]) > 0
                    and len(x["labels"]) > 0
            )
            required_cols = ["input_ids", "labels"]
            keep_cols = [c for c in train_ds.column_names if c in required_cols or c == "attention_mask"]
            train_ds = train_ds.remove_columns([c for c in train_ds.column_names if c not in keep_cols])
            
        else:
            print("Data needs tokenization, preprocessing...")
            
            # Filtering invalid samples 
            original_size = len(train_ds)
            train_ds = train_ds.filter(
                lambda x:
                    x.get("source_text") is not None
                    and x.get("target_text") is not None
                    and len(x["source_text"].strip()) > 0
                    and len(x["target_text"].strip()) > 0
                    and x.get("source_language") in SRC_LANG_MAP
                    and x.get("target_language") in TGT_LANG
            )
            print(f"After filtering: {len(train_ds)}/{original_size} samples")

            train_ds = train_ds.map(
                preprocess_text,
                desc="Tokenizing"
            )
            
            before_removal = len(train_ds)
            train_ds = train_ds.filter(lambda x: not x.get("_remove", False))
            removed = before_removal - len(train_ds)
            if removed > 0:
                print(f"Removed {removed} samples due to tokenization errors")
            
            cols_to_keep = ["input_ids", "attention_mask", "labels"]
            cols_to_remove = [c for c in train_ds.column_names if c not in cols_to_keep]
            if cols_to_remove:
                train_ds = train_ds.remove_columns(cols_to_remove)

        print(f"Final dataset size: {len(train_ds)} samples")

        train_loader = DataLoader(
            train_ds,
            batch_size=BATCH_SIZE,
            shuffle=True,
            collate_fn=data_collator,
        )

        optimizer.zero_grad()
        epoch_loss = 0.0

        for step, batch in enumerate(train_loader):
            
            batch = {k: v.to(DEVICE) for k, v in batch.items() if isinstance(v, torch.Tensor)}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
                loss = outputs.loss / GRAD_ACCUM_STEPS

            loss.backward()
            epoch_loss += loss.item() * GRAD_ACCUM_STEPS
            if (step + 1) % GRAD_ACCUM_STEPS == 0:
                
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1

                # Logging
                if global_step % 10000 == 0:
                    avg_loss = epoch_loss / (step + 1)
                    lr = scheduler.get_last_lr()[0]
                    print(f"Step {global_step:5d} | Loss: {avg_loss:.4f} | LR: {lr:.2e}")

                # Evaluation
                if global_step % 10000 == 0:
                    print(f"\n{'='*60}")
                    print(f"EVALUATION at step {global_step}")
                    print(f"{'='*60}")
                    
                    model.eval()
                    preds, refs, srcs = [], [], []

                    # Evaluate on subset of validation data
                    eval_subset = valid_data[:50]
                    
                    with torch.no_grad():
                        for ex in eval_subset:
                            tokenizer.src_lang = SRC_LANG_MAP[ex["source_language"]]
                            tgt_lang = TGT_LANG[ex["target_language"]]
                            
                            inputs = tokenizer(
                                ex["source_text"],
                                return_tensors="pt",
                                truncation=True,
                                max_length=MAX_SOURCE_LEN
                            ).to(DEVICE)

                            output = model.generate(
                                **inputs,
                                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                                max_length=MAX_TARGET_LEN,
                                num_beams=4,
                            )

                            pred_text = tokenizer.decode(output[0], skip_special_tokens=True)
                            preds.append(pred_text)
                            refs.append(ex["target_text"])
                            srcs.append(ex["source_text"])

                    # Computing metrics
                    scores = compute_metrics(preds, refs, srcs)
                    
                    print("\nValidation Metrics:")
                    for metric, score in scores.items():
                        print(f"  {metric:10s}: {score:.2f}")
                    
                    # saving translations
                    save_translations(preds, refs, srcs, "validation", global_step)
                    
                    # saving best model, here we only save a new checkpoint if the performance on the evaluation set was improved from the last checkpoint
                    if scores["BLEU"] > best_bleu:
                        best_bleu = scores["BLEU"]
                        best_path = f"{OUTPUT_DIR}/best_model"
                        model.save_pretrained(best_path)
                        tokenizer.save_pretrained(best_path)
                        print(f" New best model saved! BLEU: {best_bleu:.2f}")

                    model.train()

                # Save checkpoint
                if global_step % SAVE_STEPS == 0:
                    ckpt_path = f"{OUTPUT_DIR}/checkpoint-{global_step}"
                    model.save_pretrained(ckpt_path)
                    tokenizer.save_pretrained(ckpt_path)
                    print(f"Checkpoint saved → {ckpt_path}")

        # Epoch summary
        epoch_time = (time.time() - epoch_start) / 60
        avg_epoch_loss = epoch_loss / len(train_loader)
        print(f"\n Epoch {epoch+1} completed in {epoch_time:.1f} minutes")
        print(f"   Average loss: {avg_epoch_loss:.4f}")

    # saving final model
    final_path = f"{OUTPUT_DIR}/final_model"
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)
    print(f"Final model saved → {final_path}")

    training_time = (time.time() - training_start) / 3600
    print(f"Total time: {training_time:.2f} hours")
    print(f"Best BLEU: {best_bleu:.2f}")
    print(f"Output directory: {OUTPUT_DIR}")

    # evaluating the model on the test set
    model.eval()
    test_preds, test_refs, test_srcs = [], [], []

    with torch.no_grad():
        for ex in test_data[:100]:
            tokenizer.src_lang = SRC_LANG_MAP[ex["source_language"]]
            tgt_lang = TGT_LANG[ex["target_language"]]
            
            inputs = tokenizer(
                ex["source_text"],
                return_tensors="pt",
                truncation=True,
                max_length=MAX_SOURCE_LEN
            ).to(DEVICE)

            output = model.generate(
                **inputs,
                forced_bos_token_id=tokenizer.convert_tokens_to_ids(tgt_lang),
                max_length=MAX_TARGET_LEN,
                num_beams=5,
            )

            pred_text = tokenizer.decode(output[0], skip_special_tokens=True)
            test_preds.append(pred_text)
            test_refs.append(ex["target_text"])
            test_srcs.append(ex["source_text"])

    test_scores = compute_metrics(test_preds, test_refs, test_srcs)

    print("\nTest Set Results:")
    for metric, score in test_scores.items():
        print(f"  {metric:10s}: {score:.2f}")

    save_translations(test_preds, test_refs, test_srcs, "test", global_step)


: 

### **Finetuning Helsinki/opus-en-ar-mt**

In [ ]:
def finetune_helsinki(
    dataset_file: str = "finetune_dataset.jsonl",
    model_name: str = "Helsinki-NLP/opus-mt-en-ar",
    model_path: str = "/dev/shm/opus_en_ar_model",
    output_dir: str = "/dev/shm/opus_en_ar_output",
    translations_dir: str = "/dev/shm/opus_en_ar_translations",
    max_source_len: int = 256,
    max_target_len: int = 256,
    batch_size: int = 4,
    grad_accum_steps: int = 8,
    lr: float = 3e-4,
    epochs: int = 3,
    subset_ratio: float = 0.2,
    warmup_ratio: float = 0.1,
    save_steps: int = 1000,
    lora_r: int = 32,
    lora_alpha: int = 64,
    lora_dropout: float = 0.05,
    src_lang_map: dict = None,
    tgt_lang_map: dict = None,
):
    """
    Function to run finetuning of the Helsinki-NLP/opus-mt-en-ar model
    using the same structure as the NLLB finetune function.
    Assumes dataset_file is a JSONL file with:
      - source_text
      - target_text
      - source_language (e.g., "en")
      - target_language (e.g., "ar")
    """
    import warnings
    warnings.filterwarnings("ignore")
    import os
    os.environ["PYTHONWARNINGS"] = "ignore"

    
    HF_CACHE = "/dev/shm/hf_cache"
    os.environ["HF_HOME"] = HF_CACHE
    os.environ["TRANSFORMERS_CACHE"] = f"{HF_CACHE}/transformers"
    os.environ["HF_DATASETS_CACHE"] = f"{HF_CACHE}/datasets"
    os.environ["HF_EVALUATE_CACHE"] = f"{HF_CACHE}/evaluate"

    for path in [
        os.environ["TRANSFORMERS_CACHE"],
        os.environ["HF_DATASETS_CACHE"],
        os.environ["HF_EVALUATE_CACHE"],
        f"{HF_CACHE}/model_offload",
    ]:
        os.makedirs(path, exist_ok=True)

    # GPU optimizations
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass

    dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float16
    device = "cuda" if torch.cuda.is_available() else "cpu"


    if src_lang_map is None:
        src_lang_map = {"en": "en"}
    if tgt_lang_map is None:
        tgt_lang_map = {"ar": "ar"}

    # loading and downloading model
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(translations_dir, exist_ok=True)

    if not os.path.exists(model_path) or len(os.listdir(model_path)) < 5:
        
        snapshot_download(
            repo_id=model_name,
            local_dir=model_path,
            local_dir_use_symlinks=False,
            resume_download=True,
        )

    # tokenizer loading
    tokenizer = AutoTokenizer.from_pretrained(model_path, use_fast=False)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
        bnb_4bit_use_double_quant=True,
    )

    model = AutoModelForSeq2SeqLM.from_pretrained(
        model_path,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=dtype,
        low_cpu_mem_usage=True,
    )

    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model = prepare_model_for_kbit_training(model)

    #checking existing checkpoints
    checkpoint_dirs = glob.glob(f"{output_dir}/checkpoint-*")
    resume_checkpoint = None

    if checkpoint_dirs:
        checkpoint_dirs.sort(key=lambda x: int(x.split("-")[-1]))
        resume_checkpoint = checkpoint_dirs[-1]
        print("\n" + "=" * 60)
        print(f"📂 FOUND EXISTING CHECKPOINT: {resume_checkpoint}")
        print("=" * 60)


# lora setup

    lora_config = LoraConfig(
        task_type=TaskType.SEQ_2_SEQ_LM,
        r=lora_r,
        lora_alpha=lora_alpha,
        lora_dropout=lora_dropout,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "wi",
            "wo",
        ],
        bias="none",
    )

    if resume_checkpoint:
        print("Loading LoRA adapters from checkpoint...")
        model = PeftModel.from_pretrained(model, resume_checkpoint, is_trainable=True)
        for name, param in model.named_parameters():
            if "lora" in name.lower():
                param.requires_grad = True
    else:
        print("Initializing new LoRA adapters...")
        model = get_peft_model(model, lora_config)

    model.print_trainable_parameters()

    # data loading
    def load_jsonl(path):
        data = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                data.append(json.loads(line))
        return data

    train_data = load_jsonl(dataset_file)
    print(f"Train: {len(train_data):,} samples")

    
    # gradient fixes
    model.enable_input_require_grads()

    def make_inputs_require_grad(module, input, output):
        try:
            output.requires_grad_(True)
        except Exception:
            pass

    model.get_input_embeddings().register_forward_hook(make_inputs_require_grad)

    if model.get_output_embeddings() is not None:
        model.get_output_embeddings().register_forward_hook(make_inputs_require_grad)

    if hasattr(model, "base_model"):
        model.base_model.get_input_embeddings().register_forward_hook(
            make_inputs_require_grad
        )

    model.train()
    print("Gradient hooks applied")

    # optimizer and scheduler setup
    optimizer = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=lr,
        betas=(0.9, 0.95),
        weight_decay=0.01,
        eps=1e-8,
    )

    subset_size = max(1, int(len(train_data) * subset_ratio))
    steps_per_epoch = max(1, subset_size // (batch_size * grad_accum_steps))
    total_steps = steps_per_epoch * epochs

    print(f"Training samples per epoch: {subset_size:,}")
    print(f"Steps per epoch: {steps_per_epoch:,}")
    print(f"Total training steps: {total_steps:,}")

    scheduler = get_scheduler(
        "cosine",
        optimizer=optimizer,
        num_warmup_steps=int(warmup_ratio * total_steps),
        num_training_steps=total_steps,
    )

    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        padding=True,
        pad_to_multiple_of=8,
    )

    # starting point from last checkup calculation
    global_step = 0
    start_epoch = 0

    if resume_checkpoint:
        global_step = int(resume_checkpoint.split("-")[-1])
        start_epoch = global_step // steps_per_epoch
        for _ in range(global_step):
            scheduler.step()
        print(f"Scheduler synced to step {global_step:,}")

    # training the model
    training_start = time.time()

    for epoch in range(start_epoch, epochs):
        if subset_size > len(train_data):
            current_sample = train_data
        else:
            current_sample = random.sample(train_data, subset_size)

        sampled = []

        for item in current_sample:
            if (
                item.get("source_text")
                and item.get("target_text")
                and item.get("source_language") in src_lang_map
                and item.get("target_language") in tgt_lang_map
            ):
    
                model_inputs = tokenizer(
                    item["source_text"],
                    truncation=True,
                    max_length=max_source_len,
                )
                labels = tokenizer(
                    item["target_text"],
                    truncation=True,
                    max_length=max_target_len,
                )
                sampled.append(
                    {
                        "input_ids": model_inputs["input_ids"],
                        "attention_mask": model_inputs.get(
                            "attention_mask",
                            [1] * len(model_inputs["input_ids"]),
                        ),
                        "labels": labels["input_ids"],
                    }
                )

        train_ds = Dataset.from_list(sampled)
        print(f"Dataset ready: {len(train_ds):,} samples")

        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            collate_fn=data_collator,
            num_workers=2,
            pin_memory=True,
        )

        optimizer.zero_grad()
        epoch_loss = 0.0

        for step, batch in enumerate(train_loader):
            try:
                batch = {k: v.to(device) for k, v in batch.items()}

                with torch.cuda.amp.autocast(dtype=torch.bfloat16):
                    outputs = model(**batch)
                    loss = outputs.loss / grad_accum_steps

                loss.backward()
                epoch_loss += loss.item() * grad_accum_steps

                if (step + 1) % grad_accum_steps == 0:
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()
                    global_step += 1

                if step % 50 == 0:
                    torch.cuda.empty_cache()
                    import gc
                    gc.collect()

                if global_step % 100 == 0 and global_step > 0:
                    lr_now = scheduler.get_last_lr()[0]
                    avg_loss = epoch_loss / (step + 1)
                    print(
                        f"Step {global_step:7,d} | Loss: {avg_loss:.4f} | LR: {lr_now:.2e}"
                    )

                if (
                    global_step % save_steps == 0
                    and (step + 1) % grad_accum_steps == 0
                    and global_step > 0
                ):
                    ckpt_path = f"{output_dir}/checkpoint-{global_step}"
                    model.save_pretrained(ckpt_path)
                    tokenizer.save_pretrained(ckpt_path)
                    print(f"Checkpoint saved → {ckpt_path}")

                    all_checkpoints = sorted(
                        glob.glob(f"{output_dir}/checkpoint-*"),
                        key=lambda x: int(x.split("-")[-1]),
                    )
                    if len(all_checkpoints) > 3:
                        for old_ckpt in all_checkpoints[:-3]:
                            shutil.rmtree(old_ckpt)
                            print(f"Removed old checkpoint: {old_ckpt}")

                    torch.cuda.empty_cache()

            except RuntimeError as e:
                if "out of memory" in str(e).lower():
                    print(
                        f"OOM at step {global_step}, clearing cache and continuing"
                    )
                    torch.cuda.empty_cache()
                    import gc
                    gc.collect()
                    continue
                else:
                    raise e

        print(
            f"Epoch {epoch + 1} done | Avg loss: {epoch_loss / len(train_loader):.4f}"
        )

    # saving final model
    final_path = f"{output_dir}/final_model"
    model.save_pretrained(final_path)
    tokenizer.save_pretrained(final_path)

    print(f"Total time: {(time.time() - training_start) / 3600:.2f} hours")
    print(f"Final model: {final_path}")
    print(f"Final step: {global_step:,}")
    print("FILES IN /dev/shm - COPY TO PERMANENT STORAGE!")
    print(f"   cp -r {output_dir} /your/permanent/location/")

    return final_path



# Third, human in the loop pipeline

### **1. pre processor agent**

In [ ]:
class PreprocessorAgent:
    """
    Normalizes and structures human edits for translation corrections.
    Specialized for English/French → Arabic translation.
    """
    
    def __init__(self, source_languages: List[str] = None, target_language: str = "ar"):
        """
        Args:
            source_languages: List of ISO language codes for source languages (e.g., ['en', 'fr'])
            target_language: ISO language code for target language (default: 'ar' for Arabic)
        """
        self.source_languages = source_languages or ["en", "fr"]
        self.target_language = target_language
    
    def normalize_arabic(self, text: str) -> str:
        """
        Normalize Arabic text: remove diacritics, standardize characters and punctuation.
        """
        # Normalize Arabic letters
        arabic_replacements = {
            "أ": "ا", "إ": "ا", "آ": "ا", "ٱ": "ا",
            "ى": "ي", "ئ": "ي"
        }
        for orig, repl in arabic_replacements.items():
            text = text.replace(orig, repl)

        # Normalize Arabic punctuation
        arabic_punct = {"،": ",", "؛": ";", "؟": "?", "«": '"', "»": '"', "٫": ".", "．": "."}
        for orig, repl in arabic_punct.items():
            text = text.replace(orig, repl)

        # Normalize Arabic digits to Western digits
        arabic_digits = {"٠": "0", "١": "1", "٢": "2", "٣": "3", "٤": "4",
                         "٥": "5", "٦": "6", "٧": "7", "٨": "8", "٩": "9"}
        for orig, repl in arabic_digits.items():
            text = text.replace(orig, repl)

        # Remove emojis and symbols
        text = re.sub(r'[\U0001F600-\U0001F64F'
                      r'\U0001F300-\U0001F5FF'
                      r'\U0001F680-\U0001F6FF]', '', text)

        # Remove Arabic diacritics (tashkeel)
        arabic_diacritics = re.compile("""
                                 ّ | َ | ً | ُ | ٌ | ِ | ٍ | ْ | ـ
                             """, re.VERBOSE)
        text = re.sub(arabic_diacritics, '', text)

        # Normalize hyphens
        hyphens = ["–", "—", "ـ", "−", "_", "\u2011"]
        for h in hyphens:
            text = text.replace(h, "-")
        text = re.sub(r'-+', '-', text)

        # normalize quotes
        text = re.sub(r'[«»“”„"]', '"', text)

        return text

    def normalize_latin(self, text: str) -> str:
        """
        Normalize Latin text (French/English): standardize quotes, apostrophes, and punctuation.
        """
        # Decode escaped quotes/backslashes
        text = text.replace('\\"', '"').replace('\\\\', '\\')

        # Unicode normalization
        text = unicodedata.normalize("NFC", text)

        # Normalize apostrophes
        apostrophes = ["'", "`", "´", "ʼ"]
        for a in apostrophes:
            text = text.replace(a, "'")

        # Normalize quotation marks
        quote_map = {
            "«": '"', "»": '"',
            """: '"', """: '"', "„": '"'
        }
        for q, repl in quote_map.items():
            text = text.replace(q, repl)

        # Normalize hyphens/dashes
        hyphens = ["–", "—", "−", "-", "‒", "_", "\u2011"]
        for h in hyphens:
            text = text.replace(h, "-")
        text = re.sub(r'-+', '-', text)

        # Remove HTML tags
        text = re.sub(r'<[^>]+>', '', text)

        # Remove control characters
        text = ''.join(ch for ch in text if unicodedata.category(ch)[0] != "C")

        return text

    def normalize_whitespace_and_punctuation(self, text: str) -> str:
        """
        Normalize whitespace and remove duplicated punctuation.
        """
        # Normalize whitespace
        text = re.sub(r'\s+', ' ', text).strip()
        
        # Remove duplicated punctuation
        text = re.sub(r'([?.!,;:])\1+', r'\1', text)
        
        return text

    def normalize_text(self, text: str, is_arabic: bool = False) -> str:
        """
        Apply appropriate normalization based on language.
        
        Args:
            text: Text to normalize
            is_arabic: True if text is in Arabic, False for Latin scripts
            
        Returns:
            Normalized text
        """
        if is_arabic:
            text = self.normalize_arabic(text)
        else:
            text = self.normalize_latin(text)
        
        # Apply common normalization
        text = self.normalize_whitespace_and_punctuation(text)
        
        return text
    
    def detect_language(self, text: str) -> str:
        """
        Detect the language of the text.
        """
        try:
            return langdetect.detect(text)
        except:
            return "unknown"
    
    def process(self, source: str, llm_outputs: Dict[str, str], 
                human_edit: str) -> Optional[Dict]:
        """        
        Args:
            source: Original source text (French/English) - already normalized, passed as-is
            llm_outputs: Dictionary of LLM translations in Arabic {"model1": "arabic_translation1", ...}
            human_edit: Human-corrected translation (Arabic)
            
        Returns:
            Structured JSON object or None if validation fails
        """
        
        # Source is already normalized (French/English), use as-is
        normalized_source = source
        
        # Normalize human edit (Arabic)
        normalized_edit = self.normalize_text(human_edit, is_arabic=True)
        
        # Normalize LLM outputs (Arabic)
        normalized_llm_outputs = {}
        for model, translation in llm_outputs.items():
            normalized_llm_outputs[model] = self.normalize_text(translation, is_arabic=True)
        
        # Validate human edit is not empty
        if not normalized_edit or normalized_edit.isspace():
            print("Preprocessor: Human edit is empty after normalization")
            return None
        
        # Detect language of human edit (should be Arabic)
        detected_edit_lang = self.detect_language(normalized_edit)
        
        if detected_edit_lang != self.target_language:
            print(f" Warning: Human edit language '{detected_edit_lang}' differs from expected '{self.target_language}'")
            # Continue anyway but flag it
        
        # Build structured output
        result = {
            "source": normalized_source,
            "source_language": langdetect.detect(normalized_source),  
            "llm_outputs": normalized_llm_outputs,
            "human_edit": normalized_edit,
            "target_language": self.target_language,  # Target is Arabic
            "processed_at": datetime.now().isoformat()
        }
        
        print(f"Preprocessor: Successfully processed correction")
        
        return result

### **2. Validator agent**

1. **Exact Duplication Check**: Computes embedding similarity between human translation and all model outputs; rejects if any similarity exceeds 0.995 (near-identical copies).

2. **Semantic Correctness Verification**: Uses LLM to verify human translation preserves exact source meaning; rejects if meaning is altered (e.g., "apples" translated as "bananas").

3. **Near-Duplication Assessment**: For high similarity (≥0.92), invokes LLM to judge if human edits provide clear quality improvement over all model outputs; rejects trivial changes.

4. **Quality Improvement Check**: For lower similarity (<0.92), uses LLM to determine if human translation is clearly better than all model outputs; rejects if equal or worse.

5. **Confidence Scoring**: Computes weighted score from character-level accuracy (CHRF), length ratio, and LLM quality judgment; accepts if score ≥0.70 threshold.

6. **Special Handling**: High-similarity cases (>0.90) with good scores (>0.80) are accepted despite similarity, preventing false rejections of legitimate improvements.

In [ ]:
class ValidatorAgent:
    """
    Translation correction validator with clear decision tree.

    Decision Flow:
    1. Check for exact duplication (>0.995) → REJECT
    2. Check for semantic errors via LLM → REJECT if meaning wrong
    3. Check for near-duplication (>0.92) → Check quality
    4. Check quality improvement → REJECT if not better
    5. Compute confidence score → ACCEPT/REJECT
    """

    def __init__(
        self,
        accept_threshold: float = 0.70,
        exact_threshold: float = 0.995,
        near_threshold: float = 0.92,
        weights: dict | None = None,
        judge_model: str = "Qwen/Qwen2.5-0.5B-Instruct",
        embedding_model: str = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        model_path: str = "./models",
        device: str | None = None,
    ):
        self.accept_threshold = accept_threshold
        self.exact_threshold = exact_threshold
        self.near_threshold = near_threshold
    
        self.weights = weights or {
            "chrf": 0.4,
            "length": 0.1,
            "llm_judge": 0.5,
        }
        if abs(sum(self.weights.values()) - 1.0) > 1e-6:
            raise ValueError("Weights must sum to 1.0")
    
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.chrf_metric = CHRF()
    
        # Create models directory if it doesn't exist
        os.makedirs(model_path, exist_ok=True)
        
        # loading embedding model
        embedding_dir = os.path.join(model_path, embedding_model.replace("/", "__"))
        
        if os.path.exists(embedding_dir):
            print(f"Loading embedding model from local cache: {embedding_dir}")
            try:
                self.embedder = SentenceTransformer(embedding_dir, device=self.device)
                print("Embedding model loaded successfully from local cache")
            except Exception as e:
                print(f"Error loading local embedding model: {e}")
                raise RuntimeError(f"Failed to load embedding model from {embedding_dir}.")
        else:
            print(f"Local embedding model not found. Downloading from Hugging Face...")
            self.embedder = SentenceTransformer(embedding_model, device=self.device)
            os.makedirs(embedding_dir, exist_ok=True)
            self.embedder.save(embedding_dir)
            print(f"Model saved to local cache: {embedding_dir}")
        
        # loading judge model
        judge_dir = os.path.join(model_path, judge_model.replace("/", "__"))
        
        if not os.path.exists(judge_dir):
            print(f"Local judge model not found. Downloading from Hugging Face...")
            self.tokenizer = AutoTokenizer.from_pretrained(judge_model)
            self.model = AutoModelForCausalLM.from_pretrained(
                judge_model,
                torch_dtype=torch.float16 if self.device == "cuda" else torch.float32
            ).to(self.device)
            os.makedirs(judge_dir, exist_ok=True)
            self.model.save_pretrained(judge_dir)
            self.tokenizer.save_pretrained(judge_dir)
            print(f"Judge model saved to local cache: {judge_dir}")

                
        
        files = os.listdir(judge_dir)
        
        # checking for model files - accept either .safetensors or .bin
        has_safetensors = any(f.endswith('.safetensors') for f in files)
        has_pytorch_bin = any(f.endswith('.bin') for f in files)
        
        
        # Checking for essential files
        essential_files = ["config.json", "tokenizer.json", "tokenizer_config.json"]
        missing_essential = []
        
        for file in essential_files:
            if not os.path.exists(os.path.join(judge_dir, file)):
                missing_essential.append(file)
        
        if missing_essential:
            print(f"Missing essential files: {missing_essential}")
            raise RuntimeError(f"Missing essential files: {missing_essential}")
        
        print("All essential files present")
        
        # loading judge model
        print("\nLoading judge model (offline mode)")
        try:
            # Load tokenizer
            self.tokenizer = AutoTokenizer.from_pretrained(
                judge_dir,
                local_files_only=True,
                trust_remote_code=False 
            )
            
            # Determine dtype
            model_dtype = torch.float16 if self.device == "cuda" else torch.float32
            
            # Load model - transformers will automatically use .safetensors if available
            self.model = AutoModelForCausalLM.from_pretrained(
                judge_dir,
                torch_dtype=model_dtype,
                local_files_only=True,
                trust_remote_code=False
            ).to(self.device)
            
            print("Model loaded successfully!")
            
        except Exception as e:
            print(f"Failed to load model: {e}")
            
            # Try with trust_remote_code=True if the first attempt failed
            print("Trying with trust_remote_code=True.")
            try:
                self.tokenizer = AutoTokenizer.from_pretrained(
                    judge_dir,
                    local_files_only=True,
                    trust_remote_code=True
                )
                self.model = AutoModelForCausalLM.from_pretrained(
                    judge_dir,
                    torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                    local_files_only=True,
                    trust_remote_code=True
                ).to(self.device)
                print("Model loaded with trust_remote_code=True!")
            except Exception as e2:
                print(f"Still failed: {e2}")
                
                # Last resort: create a symbolic link if .safetensors exists but code expects .bin
                if has_safetensors and not has_pytorch_bin:
                    print("Creating symbolic link: model.safetensors → pytorch_model.bin")
                    try:
                        import sys
                        if sys.platform == "win32":
                            # Windows
                            import subprocess
                            safetensors_path = os.path.join(judge_dir, "model.safetensors")
                            bin_path = os.path.join(judge_dir, "pytorch_model.bin")
                            subprocess.run(['mklink', bin_path, safetensors_path], shell=True)
                        else:
                            # Unix/Linux/Mac
                            os.symlink(
                                os.path.join(judge_dir, "model.safetensors"),
                                os.path.join(judge_dir, "pytorch_model.bin")
                            )
                        
                        # Try loading again
                        self.tokenizer = AutoTokenizer.from_pretrained(
                            judge_dir,
                            local_files_only=True
                        )
                        self.model = AutoModelForCausalLM.from_pretrained(
                            judge_dir,
                            torch_dtype=torch.float16 if self.device == "cuda" else torch.float32,
                            local_files_only=True
                        ).to(self.device)
                    except Exception as e3:
                        raise RuntimeError(f"Failed to load model from {judge_dir}")
                else:
                    raise RuntimeError(f"Failed to load model from {judge_dir}")
        
        self.model.eval()

    
    # defining metric helpers
    def _embedding_similarity(self, a: str, b: str) -> float:
        """Compute cosine similarity between two texts."""
        ea = self.embedder.encode(a, normalize_embeddings=True)
        eb = self.embedder.encode(b, normalize_embeddings=True)
        return float(util.cos_sim(ea, eb))

    def _chrf(self, ref: str, hyp: str) -> float:
        return self.chrf_metric.sentence_score(hyp, [ref]).score / 100.0

    def _length_ratio(self, ref: str, hyp: str) -> float:
        return min(len(ref), len(hyp)) / max(len(ref), len(hyp), 1)

    # LLM judges
    def _llm_judge_meaning(self, source: str, human: str) -> float:
        """
        Verify if translation preserves source meaning.
        """
        prompt = (
            "You are a translation accuracy checker. "
            "Does the Arabic translation preserve the EXACT meaning of the English?\n\n"
            f"English: {source}\n"
            f"Arabic: {human}\n\n"
            "Check these specific things:\n"
            "1. Are key nouns the same? (e.g., if English says 'apples', Arabic should say 'تفاح' not 'موز')\n"
            "2. Is the action/verb the same?\n"
            "3. Are quantities/numbers the same?\n"
            "4. Is the time reference the same?\n"
            "5. Is the meaning preserved (not opposite)?\n\n"
            "Answer with ONLY '1' if ALL meaning is preserved correctly, or '0' if ANY part is wrong.\n"
        )
        result = self._run_judge_simple(prompt)
        return result

    def _llm_judge_quality(self, source: str, llm_outputs: Dict[str, str], human: str) -> float:
        """
        Judge if human translation is better than model outputs.
        """
        # 1. computing all similarities
        similarities = []
        for model_output in llm_outputs.values():
            sim = self._embedding_similarity(human, model_output)
            similarities.append(sim)
        
        max_sim = max(similarities) if similarities else 0
        
        prompt = (
            "Compare these translations and decide if the human one is BETTER.\n\n"
            f"Source English: {source}\n\n"
            "Machine translations:\n"
        )
        
        for i, (model_name, translation) in enumerate(llm_outputs.items(), 1):
            prompt += f"Machine {i}: {translation}\n"
        
        prompt += f"\nHuman translation: {human}\n\n"
        
        # Add strictness warning for high similarity
        if max_sim > 0.85:
            prompt += (
                "NOTE: The human translation is very similar to machine ones. "
                "Only mark as better if it fixes clear errors or is significantly more natural.\n\n"
            )
        
        prompt += (
            "Consider:\n"
            "1. Grammar correctness\n"
            "2. Natural Arabic phrasing\n"
            "3. Appropriate word choice\n"
            "4. Overall fluency\n\n"
            "Answer with '1' if human is CLEARLY better, '0' if equal or worse.\n"
            "Only '1' or '0'.\n"
        )
        
        return self._run_judge_simple(prompt)

    def _run_judge_simple(self, prompt: str) -> float:
        """Simplified judge runner."""
        inputs = self.tokenizer(
            prompt, 
            return_tensors="pt", 
            truncation=True, 
            max_length=1024  
        ).to(self.device)
        
        with torch.no_grad():
            output = self.model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False,
                pad_token_id=self.tokenizer.eos_token_id,
                temperature=0.1,
            )
        
        decoded = self.tokenizer.decode(output[0], skip_special_tokens=True).strip()
        
        # Look for 1 or 0 in the output
        if "1" in decoded and "0" not in decoded[-10:]:  
            return 1.0
        elif "0" in decoded and "1" not in decoded[-10:]:
            return 0.0
        else:
            # Default to 0 if unclear
            return 0.0

    # checking for quality imporvement
    def _check_quality_improvement(self, human: str, llm_outputs: Dict[str, str]) -> float:
        """
        Check if human translation shows quality improvement.
        Returns score from 0-1.
        """
        # 1. Check if human is longer than all models (often indicates improvement)
        human_len = len(human)
        max_model_len = max(len(t) for t in llm_outputs.values())
        
        if human_len > max_model_len * 1.3:  # 30% longer
            length_score = 0.7
        elif human_len > max_model_len:
            length_score = 0.5
        else:
            length_score = 0.2
        
        # 2. Check CHRF improvement
        chrf_scores = []
        for model_output in llm_outputs.values():
            # Human compared to model (reverse of usual)
            score = self._chrf(model_output, human)
            chrf_scores.append(score)
        
        avg_chrf = sum(chrf_scores) / len(chrf_scores) if chrf_scores else 0
        chrf_score = min(avg_chrf * 2, 1.0)  # Scale up
        
        # 3. Combine scores
        return (length_score * 0.3 + chrf_score * 0.7)

    
    def validate(
        self,
        source_text: str,
        language_pair: Tuple[str, str],
        llm_outputs: Dict[str, str],
        human_translation: str,
    ) -> Dict:
        """
        Validates a human translation against LLM outputs.
        """
        # Get similarity scores
        sim_to_models = [
            self._embedding_similarity(human_translation, t)
            for t in llm_outputs.values()
        ]
        max_sim = max(sim_to_models) if sim_to_models else 0

        # 1. check for exact duplication
        if max_sim >= self.exact_threshold:
            return {
                "source_text": source_text,
                "language-pair": f"{language_pair[0]}-{language_pair[1]}",
                "llm_outputs": llm_outputs,
                "human_translation": human_translation,
                "score": 0.0,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "decision": "reject",
                "reason": f"human edit too similar to model output (similarity={max_sim:.3f})",
            }

        # 2. check for semantic correctness
        meaning_preserved = self._llm_judge_meaning(source_text, human_translation)
        
        if meaning_preserved < 0.5:  
            return {
                "source_text": source_text,
                "language-pair": f"{language_pair[0]}-{language_pair[1]}",
                "llm_outputs": llm_outputs,
                "human_translation": human_translation,
                "score": 0.0,
                "timestamp": datetime.now(timezone.utc).isoformat(),
                "decision": "reject",
                "reason": "translation may not preserve source meaning",
            }

        # 3. check for near dup and quality
        if max_sim >= self.near_threshold:
            # For high similarity, we need to check quality
            is_better = self._llm_judge_quality(source_text, llm_outputs, human_translation)
            
            if is_better < 0.5:  # Not clearly better
                # Additional check: maybe it's still good enough
                quality_score = self._check_quality_improvement(human_translation, llm_outputs)
                if quality_score < 0.6:  # Not enough improvement
                    return {
                        "source_text": source_text,
                        "language-pair": f"{language_pair[0]}-{language_pair[1]}",
                        "llm_outputs": llm_outputs,
                        "human_translation": human_translation,
                        "score": quality_score,
                        "timestamp": datetime.now(timezone.utc).isoformat(),
                        "decision": "reject",
                        "reason": f"high similarity ({max_sim:.3f}) with insufficient improvement",
                    }
                else:
                    # Has some improvement, continue to scoring
                    is_better = 1.0
        else:
            # Lower similarity, just check if it's better
            is_better = self._llm_judge_quality(source_text, llm_outputs, human_translation)
            
            if is_better < 0.5:
                quality_score = self._check_quality_improvement(human_translation, llm_outputs)
                if quality_score < 0.5:
                    return {
                        "source_text": source_text,
                        "language-pair": f"{language_pair[0]}-{language_pair[1]}",
                        "llm_outputs": llm_outputs,
                        "human_translation": human_translation,
                        "score": quality_score,
                        "timestamp": datetime.now(timezone.utc).isoformat(),
                        "decision": "reject",
                        "reason": "not better than model outputs",
                    }
                else:
                    is_better = 1.0

        # 4. computing final score
        # Get best CHRF score (human vs models)
        chrf_score = max(self._chrf(t, human_translation) for t in llm_outputs.values())
        
        # Get best length ratio
        length_score = max(
            self._length_ratio(t, human_translation) 
            for t in llm_outputs.values()
        )
        
        # Adjust is_better based on similarity
        if max_sim > 0.85 and is_better > 0.5:
            # For high similarity, require stronger evidence of improvement
            quality_improvement = self._check_quality_improvement(human_translation, llm_outputs)
            is_better = quality_improvement  # Use the computed improvement score

        metrics = {
            "chrf": chrf_score,
            "length": length_score,
            "llm_judge": float(is_better),
        }

        final_score = sum(metrics[k] * self.weights[k] for k in self.weights)
        
        # Special case: if similarity is high (>0.9) but score is good, still accept
        if max_sim > 0.9 and final_score > 0.8:
            decision = "accept"
        else:
            decision = "accept" if final_score >= self.accept_threshold else "reject"

        result = {
            "source_text": source_text,
            "language-pair": f"{language_pair[0]}-{language_pair[1]}",
            "llm_outputs": llm_outputs,
            "human_translation": human_translation,
            "score": round(float(final_score), 4),
            "timestamp": datetime.now(timezone.utc).isoformat(),
            "decision": decision,
        }
        
        if decision == "reject":
            if final_score < self.accept_threshold:
                result["reason"] = f"confidence score below threshold ({final_score:.3f} < {self.accept_threshold})"
            elif max_sim > self.near_threshold:
                result["reason"] = f"high similarity ({max_sim:.3f}) with insufficient improvement"
            else:
                result["reason"] = "not better than model outputs"
        
        return result

### **Deduplicator agent**

The deduplication agent keeps a vector database of all accepted human translations and uses embeddings plus FAISS to detect near‑duplicate sentences. For each new human edit, it splits the text into sentences, checks their similarity against stored ones (obtained now from the human edit or already present in the master.jsonl) and rejects the whole correction if any sentence is above 0.85 similarity.

In [ ]:
class DeduplicatorAgent:
    """
    FAISS-based deduplicator for human corrections.

    Role:
    - Maintain a vector DB of all accepted human sentences.
    - For each new human_edit, split into sentences, embed, and check
      similarity with existing embeddings.
    - If any sentence is too similar (>= threshold) to a stored one,
      mark the whole correction as duplicate and reject.
    - Otherwise, accept and update FAISS with the new sentence embeddings.
    - Initialize FAISS memory with existing entries in master.jsonl to prevent
      duplicates in fine-tuning queue.
    """

    def __init__(
        self,
        embedding_model: SentenceTransformer,
        similarity_threshold: float = 0.85,
        faiss_index_path: str | None = None,
        master_jsonl_path: str | None = None,
    ):
        """
        Args:
        embedding_model: A SentenceTransformer instance (re-use validator.embedder).
        similarity_threshold: Cosine similarity threshold for considering
        two sentences duplicates.
        faiss_index_path: Optional path to persist / load FAISS index.
        master_jsonl_path: Optional path to master.jsonl to preload FAISS memory.
        """
        self.embedder = embedding_model
        self.similarity_threshold = similarity_threshold
        self.faiss_index_path = faiss_index_path

        # storing L2-normalized embeddings to approximate cosine similarity
        self.index: faiss.Index | None = None
        self.dimension: int | None = None
        self.num_vectors: int = 0

        self._vectors: list[np.ndarray] = []

        # loading existing FAISS index if it exists
        if self.faiss_index_path and os.path.exists(self.faiss_index_path):
            self._load_index()

        # loading human corrections from master.jsonl into FAISS 
        if master_jsonl_path and os.path.exists(master_jsonl_path):
            self._load_master_into_faiss(master_jsonl_path)

    
    def _build_index(self, dim: int) -> None:
        """Initialize a FAISS index for inner product (cosine on normalized vecs)."""
        self.dimension = dim
        self.index = faiss.IndexFlatIP(dim)

    def _normalize(self, vecs: np.ndarray) -> np.ndarray:
        """L2-normalize vectors along last dimension."""
        norms = np.linalg.norm(vecs, axis=1, keepdims=True) + 1e-12
        return vecs / norms

    def _add_vectors(self, vecs: np.ndarray) -> None:
        """Add vectors to FAISS index (create if needed)."""
        if vecs.size == 0:
            return

        vecs = self._normalize(vecs).astype("float32")

        if self.index is None:
            self._build_index(vecs.shape[1])

        self.index.add(vecs)
        self.num_vectors += vecs.shape[0]
        self._vectors.append(vecs)

    def _search(self, vecs: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Search nearest neighbors for given vectors."""
        if self.index is None or self.num_vectors == 0:
            n = vecs.shape[0]
            return (
                np.zeros((n, 0), dtype="float32"),
                np.zeros((n, 0), dtype="int64"),
            )

        vecs = self._normalize(vecs).astype("float32")
        distances, indices = self.index.search(vecs, k=1)
        return distances, indices

    def _save_index(self) -> None:
        if not self.faiss_index_path or self.index is None:
            return
        faiss.write_index(self.index, self.faiss_index_path)

    def _load_index(self) -> None:
        self.index = faiss.read_index(self.faiss_index_path)
        self.dimension = self.index.d

    def _load_master_into_faiss(self, master_jsonl_path: str) -> None:
        """Preload FAISS memory with human_translation sentences from master.jsonl (batched)."""
        texts: list[str] = []

        with open(master_jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                try:
                    item = json.loads(line)
                except json.JSONDecodeError:
                    continue
                human_text = item.get("human_translation", "").strip()
                if not human_text:
                    continue
                
                sentences = self._segment_sentences(human_text)
                texts.extend(sentences)

        if not texts:
            return

        # encoding all sentences at once
        emb = self._embed_sentences(texts)
        if emb.size == 0:
            return

        self._add_vectors(emb)
        self._save_index()
        print(
            f"Deduplicator: Preloaded {emb.shape[0]} sentence embeddings from {master_jsonl_path} "
            f"into FAISS (total={self.num_vectors})"
        )

    
    def _segment_sentences(self, text: str) -> list[str]:
        """Split text into sentences."""
        parts = re.split(r"[\.!\?؟]+", text)
        sentences = [s.strip() for s in parts if s.strip()]
        return sentences

    def _embed_sentences(self, sentences: list[str]) -> np.ndarray:
        """Encode sentences into embeddings using the shared embedder."""
        if not sentences:
            return np.zeros((0, self.dimension or 384), dtype="float32")

        emb = self.embedder.encode(
            sentences,
            convert_to_numpy=True,
            normalize_embeddings=False,
        )
        if isinstance(emb, list):
            emb = np.array(emb)
        return emb.astype("float32")

    def is_duplicate(self, human_edit: str) -> bool:
        """Check if any sentence in human_edit is too similar to existing ones."""
        sentences = self._segment_sentences(human_edit)
        if not sentences:
            return False

        sent_emb = self._embed_sentences(sentences)
        if self.index is None or self.num_vectors == 0:
            return False

        distances, _ = self._search(sent_emb)
        if distances.size == 0:
            return False

        max_sim = float(distances.max())
        if max_sim >= self.similarity_threshold:
            print(
                f"Deduplicator: Found duplicate sentence (similarity={max_sim:.3f} ≥ {self.similarity_threshold})"
            )
            return True

        return False

    def update_memory(self, human_edit: str) -> None:
        """Add sentences of accepted human_edit to FAISS index."""
        sentences = self._segment_sentences(human_edit)
        if not sentences:
            return

        sent_emb = self._embed_sentences(sentences)
        self._add_vectors(sent_emb)
        self._save_index()
        print(
            f"Deduplicator: Added {sent_emb.shape[0]} sentence embeddings to FAISS (total={self.num_vectors})"
        )

    def process(self, correction: dict) -> dict:
        """Process a validated correction: reject if duplicate, else accept."""
        human_text = correction.get("human_translation", "").strip()
        if not human_text:
            return correction

        if correction.get("decision") != "accept":
            return correction

        if self.is_duplicate(human_text):
            correction["decision"] = "reject"
            reason = correction.get("reason", "")
            if reason:
                correction["reason"] = reason + "; duplicate human correction"
            else:
                correction["reason"] = "duplicate human correction"
            print("Deduplicator: Correction marked as duplicate -> reject")
            return correction

        self.update_memory(human_text)
        print("Deduplicator: Correction is unique -> keep as accept")
        return correction


### **Executor agent**

In [ ]:
class ExecutorAgent:
    """
    Persist validated and deduplicated human translation corrections to disk.
    """

    def __init__(self, output_path: str = "master.jsonl"):
        self.output_path = output_path

        # JSON schema for validation
        self.schema = {
            "type": "object",
            "properties": {
                "source_text": {"type": "string"},
                "language-pair": {"type": "string"},  
                "llm_outputs": {"type": "object"},
                "human_translation": {"type": "string"},
                "score": {"type": "number"},
                "timestamp": {"type": "string"}
            },
            "required": ["source_text", "language-pair", "llm_outputs",
                         "human_translation", "score", "timestamp"]
        }

        # Create file if it doesn't exist
        if not os.path.exists(self.output_path):
            open(self.output_path, "a", encoding="utf-8").close()

    def persist(self, correction: Dict) -> bool:
        """
        Persist a validated correction to JSONL.
        Returns True if appended, False if schema validation fails.
        """

        # Validate schema
        try:
            validate(instance=correction, schema=self.schema)
            if not correction.get("decision") == "accept":
                print("Executor: Correction not accepted, skipping persistence")
                return False
            else:
                # Remove decision and reason before saving
                correction.pop("decision", None)
                correction.pop("reason", None)
        except ValidationError as e:
            print(f"Schema validation failed: {e.message}")
            return False

        # Append to JSONL
        with open(self.output_path, "a", encoding="utf-8") as f:
            f.write(json.dumps(correction, ensure_ascii=False) + "\n")

        return True


### **Monitor Agent**

The agent that monitors and checks wether the finetuning should be done or not yet, it bases the decision off of the threshold of lines and the number of days since the finetuning last happened.

In [ ]:
class MonitorAgent:
    """
    Monitor Agent

    Role:
    - Observe accumulation of accepted, unique corrections in master.jsonl.
    - Trigger fine-tuning when thresholds are met:
      * Number of corrections (e.g., 10_000)
      * Time-based threshold since last reset (e.g., 7 days)
    - Manage lifecycle:
      * Lock file and prepare dataset for fine-tuning
      * Provide few-shot examples (last N corrections) for prompting
      * After fine-tuning, reset:
        - Clear FAISS memory (deduplicator)
        - Archive master.jsonl
        - Start fresh for next cycle
    """

    def __init__(
        self,
        path_file_jsonl: str = "master.jsonl",
        deduplicator: Optional[DeduplicatorAgent] = None,
        line_threshold: int = 10_000,
        days_threshold: int = 7,
        few_shot_n: int = 50,
        archive_dir: str = "archive",
        faiss_index_path: str = "dedup_index.faiss",
        state_path: str = "monitor_state.json",
    ):
        """
        Args:
        path_file_jsonl: Path to the master JSONL file.
        deduplicator: The DeduplicatorAgent instance (to clear FAISS).
        line_threshold: Number of corrections to trigger fine-tuning.
        days_threshold: Number of days to trigger fine-tuning.
        few_shot_n: How many latest corrections to use for few-shot.
        archive_dir: Directory where archived master.jsonl copies are stored.
        faiss_index_path: Path of FAISS index file to remove on reset.
        state_path: Path to a small JSON file storing last reset timestamp.
        """
        self.path_file_jsonl = path_file_jsonl
        self.deduplicator = deduplicator
        self.line_threshold = line_threshold
        self.days_threshold = days_threshold
        self.few_shot_n = few_shot_n
        self.archive_dir = archive_dir
        self.faiss_index_path = faiss_index_path
        self.state_path = state_path

        os.makedirs(self.archive_dir, exist_ok=True)

        # avoiding race conditions
        self._lock = threading.Lock()
        self.last_reset_ts = self._load_last_reset_timestamp()

    
    def _load_last_reset_timestamp(self) -> datetime:
        """Load last reset timestamp from a small JSON state file."""
        if not os.path.exists(self.state_path):
            ts = datetime.now()
            self._save_last_reset_timestamp(ts)
            return ts

        try:
            with open(self.state_path, "r", encoding="utf-8") as f:
                data = json.load(f)
            ts_str = data.get("last_reset_ts")
            if not ts_str:
                raise ValueError("Missing last_reset_ts")
            return datetime.fromisoformat(ts_str)
        except Exception:
            
            ts = datetime.now()
            self._save_last_reset_timestamp(ts)
            return ts

    def _save_last_reset_timestamp(self, ts: datetime) -> None:
        """Persist last reset timestamp to JSON."""
        data = {"last_reset_ts": ts.isoformat()}
        with open(self.state_path, "w", encoding="utf-8") as f:
            json.dump(data, f, ensure_ascii=False, indent=2)

    # getting the lines and time of last time master.jsonl
    def _get_line_count(self) -> int:
        """Return how many lines (corrections) are in master.jsonl."""
        if not os.path.exists(self.path_file_jsonl):
            return 0
        with open(self.path_file_jsonl, "r", encoding="utf-8") as f:
            return sum(1 for _ in f)

    def _time_since_last_reset(self) -> timedelta:
        """Return timedelta since last reset."""
        return datetime.now() - self.last_reset_ts

    # giving it the few shots example
    def get_last_n_corrections(self, n: Optional[int] = None) -> List[Dict]:
        """
        Read the last N JSON objects from master.jsonl.

        NOTE: Simple implementation: read all lines, slice last N.
        This is fine for moderate sizes; can be optimized later.
        """
        n = n or self.few_shot_n
        if not os.path.exists(self.path_file_jsonl):
            return []

        with open(self.path_file_jsonl, "r", encoding="utf-8") as f:
            lines = f.readlines()

        last_lines = lines[-n:]
        corrections = []
        for line in last_lines:
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
                corrections.append(obj)
            except json.JSONDecodeError:
                continue
        return corrections

    def build_few_shot_prompt(
        self,
        source_text: str,
        model_translation: str,
    ) -> str:
        """
        Build a few-shot prompt using last N corrections + new sentence to translate.

        Format for each example:
        Source: ...
        LLM Translation: ...
        Corrected: ...

        Then append:
        Source: 
        LLM Translation: 
        Corrected:
        """
        examples = self.get_last_n_corrections()
        parts: List[str] = []

        for ex in examples:
            src = ex.get("source_text") or ex.get("source") or ""
            llm_outputs = ex.get("llm_outputs", {})
            
            llm_translation = ""
            if isinstance(llm_outputs, dict) and llm_outputs:
                
                first_key = sorted(llm_outputs.keys())[0]
                llm_translation = llm_outputs[first_key]
            corrected = ex.get("human_translation") or ""

            parts.append(
                f"Source: {src}\n"
                f"LLM Translation: {llm_translation}\n"
                f"Corrected: {corrected}\n"
            )

        # appending corrections
        parts.append(
            f"Source: {source_text}\n"
            f"LLM Translation: {model_translation}\n"
            f"Corrected:"
        )

        return "\n".join(parts)

    # checking whether finetuning should be triggered or not yet
    def should_trigger_finetune(self) -> bool:
        """
        Decide whether fine-tuning should be triggered based on:
        - line_threshold (number of corrections)
        - days_threshold (time since last reset)
        """
        line_count = self._get_line_count()
        elapsed = self._time_since_last_reset()

        print(
            f"Monitor: master.jsonl has {line_count} lines, "
            f"time since last reset: {elapsed.days} days"
        )

        if line_count >= self.line_threshold:
            print(
                f"Monitor: Line threshold reached "
                f"({line_count} >= {self.line_threshold})"
            )
            return True

        if elapsed >= timedelta(days=self.days_threshold):
            print(
                f"Monitor: Time threshold reached "
                f"({elapsed.days} >= {self.days_threshold} days)"
            )
            return True

        return False

    # finetuning prepaeation and reset after
    def prepare_finetune_dataset(self, output_path: str = "finetune_dataset.jsonl") -> str:
        """
        Lock master.jsonl and copy it to a temporary location for fine-tuning.

        Returns:
        Path of the dataset file to use for fine-tuning.
        """
        with self._lock:
            if not os.path.exists(self.path_file_jsonl):
                print("Monitor: No master.jsonl found, nothing to prepare.")
                return ""

            shutil.copy2(self.path_file_jsonl, output_path)
            print(f"Monitor: Prepared fine-tuning dataset at {output_path}")
            return output_path

    def _reset_faiss_memory(self) -> None:
        """
        Clear FAISS memory used by the DeduplicatorAgent.
        Simple strategy: reinitialize the index and delete the index file.
        """
        if self.deduplicator is None:
            print("Monitor: No deduplicator provided, skipping FAISS reset.")
            return

        # Reset in-memory index
        self.deduplicator.index = None
        self.deduplicator.dimension = None
        self.deduplicator.num_vectors = 0
        self.deduplicator._vectors = []

        # Remove on-disk index file, if present
        if self.faiss_index_path and os.path.exists(self.faiss_index_path):
            os.remove(self.faiss_index_path)
            print(f"Monitor: Removed FAISS index file {self.faiss_index_path}")

        print("Monitor: FAISS memory cleared.")

    def archive_and_reset(self) -> str:
        """
        Archive current master.jsonl, clear FAISS memory, and start fresh.

        Returns:
        Path of the archived file.
        """
        with self._lock:
            if not os.path.exists(self.path_file_jsonl):
                print("Monitor: No master.jsonl to archive.")
                return ""

            # archiving the master for the new cycle
            ts = datetime.now().strftime("%Y%m%d_%H%M%S")
            archive_path = os.path.join(
                self.archive_dir, f"master_{ts}.jsonl"
            )
            shutil.move(self.path_file_jsonl, archive_path)
            print(f"Monitor: Archived master.jsonl to {archive_path}")

            # resetting FAISS for a new cycle
            self._reset_faiss_memory()

            # Update last reset timestamp
            self.last_reset_ts = datetime.now()
            self._save_last_reset_timestamp(self.last_reset_ts)

            # create a new master.jsonl
            open(self.path_file_jsonl, "w", encoding="utf-8").close()
            print("Monitor: Created new empty master.jsonl")

            
            if self.deduplicator is not None:
                
                embedder = self.deduplicator.embedder
                sim_thr = self.deduplicator.similarity_threshold
                self.deduplicator = DeduplicatorAgent(
                    embedding_model=embedder,
                    similarity_threshold=sim_thr,
                    faiss_index_path=self.faiss_index_path,
                    master_jsonl_path=self.path_file_jsonl,
                )
                print("Monitor: Re-instantiated DeduplicatorAgent for new cycle.")

            return archive_path

    
    def check_and_handle_cycle(self) -> Optional[str]:
        """
        High-level call to be used periodically (or after each batch):
        - Check thresholds
        - If triggered:
          * prepare dataset
          * (you run fine-tuning externally on that dataset)
          * archive and reset after fine-tuning

        Returns:
        Path to the dataset file if fine-tuning should start,
        None otherwise.
        """
        if not self.should_trigger_finetune():
            return None

        # Step 1: prepare dataset
        dataset_path = self.prepare_finetune_dataset("finetune_dataset.jsonl")

        if not dataset_path:
            return None

        print(
            "Monitor: Fine-tuning should be started externally using "
            f"dataset at {dataset_path}"
        )
        
        return dataset_path


### **Orchestrator Agent**

The agent that defines the whole pipeline, it takes the input data from the frontend after the user submits the correction. It initializes different agents from the preprocessor until the monitor and makes the data go through all of the steps. To end up at the end with ready data to be used for the fintuning if needed


In [ ]:
class OrchestratorAgent:
    """
    Orchestrator Agent

    Role:
    - Initialize and hold all pipeline agents (Preprocessor, Validator,
      Deduplicator, Executor, Monitor).
    - Expose a simple process_batch(frontend_payload) method that takes
      a list of dicts of the form:
        [
            {
                "source": "...",
                "llm_outputs": {...},
                "human_edit": "...",
            },
            ...
        ]
    - For each item:
      1) Preprocess (normalize/structure)
      2) Validate (quality/meaning)
      3) Deduplicate (FAISS)
      4) Persist accepted corrections to master.jsonl
      5) Optionally check Monitor to see if fine-tuning should start
    """

    def __init__(
        self,
        path_file_jsonl: str = "master.jsonl",
        faiss_index_path: str = "dedup_index.faiss",
        archive_dir: str = "archive",
        monitor_state_path: str = "monitor_state.json",
        line_threshold: int = 5000, # setting the limit of finetuning sentences
        days_threshold: int = 7,
        few_shot_n: int = 50,
    ):
        # Initializing the agents
        self.path_file_jsonl = path_file_jsonl

        self.preprocessor = PreprocessorAgent()
        self.validator = ValidatorAgent()

        
        self.deduplicator = DeduplicatorAgent(
            embedding_model=self.validator.embedder,
            similarity_threshold=0.85,
            faiss_index_path=faiss_index_path,
            master_jsonl_path=self.path_file_jsonl,
        )

        self.executor = ExecutorAgent(self.path_file_jsonl)

        self.monitor = MonitorAgent(
            path_file_jsonl=self.path_file_jsonl,
            deduplicator=self.deduplicator,
            line_threshold=line_threshold,
            days_threshold=days_threshold,
            few_shot_n=few_shot_n,
            archive_dir=archive_dir,
            faiss_index_path=faiss_index_path,
            state_path=monitor_state_path,
        )

    def process_batch(
        self,
        frontend_payload: list[dict],
        language_pair: tuple[str, str] = ("en", "ar"),
        check_monitor: bool = True,
    ) -> dict:
        """
        Run the full pipeline on a batch of changes coming from the frontend.

        Args:
        - frontend_payload: list of dicts with keys:
            "source": str
            "llm_outputs": dict[str, str]
            "human_edit": str
            (optional) "expected": "accept"/"reject" (used only for testing)
        - language_pair: translation direction, default ("en", "ar").
        - check_monitor: whether to call monitor.check_and_handle_cycle() after batch.

        Returns:
        - summary dict with:
            "results": list of per-item outputs (corrections + optional test info)
            "dataset_path": path to finetune dataset if monitor fired, else None
        """
        results: list[dict] = []
        any_accepted = False  

        for i, item in enumerate(frontend_payload, 1):
            source = item.get("source", "")
            llm_outputs = item.get("llm_outputs", {})
            human_edit = item.get("human_edit", "")
            expected = item.get("expected")  # may be None

            print(f"\n================ Orchestrator item {i} =================")
            print(f"Source: {source}")
            print(f"Human edit: {human_edit}")

            try:
                # 1. Preprocess
                processed = self.preprocessor.process(source, llm_outputs, human_edit)
                if processed is None:
                    print("Preprocessor: Processing failed, skipping item.")
                    results.append(
                        {
                            "index": i,
                            "status": "preprocess_failed",
                            "reason": "Preprocessor returned None",
                            "expected": expected,
                        }
                    )
                    continue

                # 2. Validate
                correction = self.validator.validate(
                    source_text=processed["source"],
                    language_pair=language_pair,
                    llm_outputs=processed["llm_outputs"],
                    human_translation=processed["human_edit"],
                )

                # 3. Deduplicate AFTER validation
                correction = self.deduplicator.process(correction)

                print("Orchestrator: Validation + deduplication output:")
                print(correction)

                # 4. Persist to JSONL
                self.executor.persist(correction)

               
                if correction.get("decision") == "accept":
                    any_accepted = True


                results.append(
                    {
                        "index": i,
                        "correction": correction,
                        "expected": expected,
                    }
                )

            except Exception as e:
                print(f"Orchestrator: Error processing item {i}: {e}")
                results.append(
                    {
                        "index": i,
                        "status": "error",
                        "reason": str(e),
                        "expected": expected,
                    }
                )

        dataset_path = None
        if check_monitor:
            if any_accepted:
                dataset_path = self.monitor.check_and_handle_cycle()
                if dataset_path:
                    print(
                        f"\nMonitor suggests starting fine-tuning with dataset: {dataset_path}"
                    )
                    # finetuning NLLB 3.3B
                    finetune_nllb_33b()
                    print("Fine-tuning of NLLB 3.3B done")

                    finetune_nllb_13b()
                    print("Fine-tuning of NLLB 1.3B done")

                    # resetting the cycle
                    self.monitor.archive_and_reset()
            else:
                # No correction was accepted in the whole batch
                dataset_path = "no correction was accepted"

        return {"results": results, "dataset_path": dataset_path}
